# Сравнение DCGAN и Stable Diffusion для дерматоскопических изображений

Ноутбук покрывает полный цикл: загрузка данных с Kaggle, подготовка датасета, обучение DCGAN, генерация через Stable Diffusion, расчёт метрик качества и формулировка выводов.

## Структура ноутбука
1. Подготовка окружения
2. Загрузка и исследование данных
3. Модель DCGAN и обучение
4. Генерация и метрики для DCGAN
5. Генерация через Stable Diffusion
6. Сравнение метрик и аналитический вывод

In [ ]:
# Установка зависимостей (выполняется один раз)
!pip install --quiet kaggle tensorflow tensorflow-addons matplotlib seaborn pillow scikit-image diffusers transformers accelerate torch torchvision

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow import keras
from tensorflow.keras import layers

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (8, 6), "axes.titlesize": 14})

print(f"TensorFlow version: {tf.__version__}")

## Подготовка и загрузка датасета

1. Зарегистрируйтесь на Kaggle и создайте токен в профиле (Account → Create API Token).
2. Сохраните файл `kaggle.json` в директорию `~/.kaggle/` и выдайте права `chmod 600 ~/.kaggle/kaggle.json`.
3. Заполните переменные окружения `KAGGLE_USERNAME` и `KAGGLE_KEY`, чтобы запуск `kaggle` CLI был доступен из ноутбука.
4. Выполните ячейки ниже, чтобы скачать и распаковать датасет `skin-cancer9-classesisic`.

In [ ]:
DATASET_SLUG = "nodoubttome/skin-cancer9-classesisic"
RAW_DATA_DIR = Path("data/raw")
EXTRACTED_DIR = RAW_DATA_DIR / "skin-cancer9-classesisic"
TRAIN_SPLIT_NAME = "Train"
VAL_SPLIT_NAME = "Test"  # в датасете test используется как holdout
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Каталог для данных: {EXTRACTED_DIR.resolve()}")

In [ ]:
import subprocess
import zipfile


def download_dataset(slug: str, target_dir: Path) -> None:
    archive_path = target_dir / f"{slug.split('/')[-1]}.zip"
    if any(target_dir.glob("**/*.jpg")):
        print("Изображения уже распакованы, пропускаем скачивание.")
        return

    print("Скачиваем архив с Kaggle...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(target_dir), "--unzip"],
        check=True,
    )
    print("Архив скачан и распакован.")


download_dataset(DATASET_SLUG, EXTRACTED_DIR)

In [ ]:
def find_split_dir(root: Path, split_name: str) -> Path:
    candidates = [p for p in root.rglob(split_name) if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"Не найден подкаталог {split_name} внутри {root}")
    # Берём самый верхний уровень
    return min(candidates, key=lambda p: len(p.parts))


train_dir = find_split_dir(EXTRACTED_DIR, TRAIN_SPLIT_NAME)
val_dir = find_split_dir(EXTRACTED_DIR, VAL_SPLIT_NAME)
print(f"Train dir: {train_dir}")
print(f"Val/Test dir: {val_dir}")

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Классов: {num_classes}, размер train: {train_ds.cardinality().numpy()} батчей")

In [ ]:
normalization_layer = layers.Rescaling(scale=1.0 / 127.5, offset=-1)
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)

def prepare(ds: tf.data.Dataset, training: bool = True) -> tf.data.Dataset:
    autotune = tf.data.AUTOTUNE
    ds = ds.map(lambda x, _: (data_augmentation(x) if training else x), num_parallel_calls=autotune)
    ds = ds.map(lambda x, _: normalization_layer(x), num_parallel_calls=autotune)
    return ds.cache().shuffle(1024, seed=SEED).prefetch(autotune)

train_images = prepare(train_ds)
val_images = prepare(val_ds, training=False)

In [ ]:
def dataset_to_dataframe(ds: tf.data.Dataset, class_labels: List[str]) -> pd.DataFrame:
    labels = []
    for _, batch_labels in ds.take(1000):  # ограничимся первыми 1000 батчей
        labels.extend(np.argmax(batch_labels.numpy(), axis=1))
    label_names = [class_labels[idx] for idx in labels]
    return pd.DataFrame({"label": label_names})

train_df = dataset_to_dataframe(train_ds, class_names)
plt.figure(figsize=(10, 4))
sns.countplot(data=train_df, x="label", order=class_names)
plt.xticks(rotation=45, ha="right")
plt.title("Распределение классов в обучающей выборке")
plt.tight_layout()
plt.show()

In [ ]:
def show_batch(ds: tf.data.Dataset, title: str):
    images, labels = next(iter(ds.unbatch().batch(16)))
    plt.figure(figsize=(10, 10))
    for i, (img, label) in enumerate(zip(images, labels)):
        plt.subplot(4, 4, i + 1)
        plt.imshow((img.numpy() + 1) / 2)
        plt.title(class_names[int(tf.argmax(label))], fontsize=8)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_batch(train_images, "Примеры после аугментаций")

## DCGAN: архитектура и обучение

Генератор преобразует 100-мерный латентный вектор в изображение 128×128, дискриминатор отличает реальные снимки от синтетических. Обучаем модели соревновательно, чередуя шаги оптимизации.

In [ ]:
LATENT_DIM = 100


def build_generator(latent_dim: int = LATENT_DIM) -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(latent_dim,)),
            layers.Dense(8 * 8 * 512, use_bias=False),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Reshape((8, 8, 512)),
            layers.Conv2DTranspose(256, 5, strides=2, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Conv2DTranspose(3, 5, strides=2, padding="same", activation="tanh"),
        ],
        name="generator",
    )
    return model


def build_discriminator() -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
            layers.Conv2D(64, 5, strides=2, padding="same"),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Conv2D(128, 5, strides=2, padding="same"),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Conv2D(256, 5, strides=2, padding="same"),
            layers.LeakyReLU(0.2),
            layers.Dropout(0.3),
            layers.Flatten(),
            layers.Dense(1),
        ],
        name="discriminator",
    )
    return model


generator = build_generator()
discriminator = build_discriminator()
generator.summary()
discriminator.summary()

In [ ]:
class DCGAN(keras.Model):
    def __init__(self, discriminator: keras.Model, generator: keras.Model, latent_dim: int):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.d_accuracy = keras.metrics.BinaryAccuracy(name="d_accuracy")

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        generated_images = self.generator(random_latent_vectors)

        combined_images = tf.concat([generated_images, real_images], axis=0)
        labels = tf.concat(
            [tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0
        )
        labels += 0.05 * tf.random.uniform(tf.shape(labels))  # label smoothing

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))
        self.d_accuracy.update_state(labels, predictions)

        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        misleading_labels = tf.ones((batch_size, 1))
        with tf.GradientTape() as tape:
            fake_images = self.generator(random_latent_vectors)
            predictions = self.discriminator(fake_images)
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss, "d_accuracy": self.d_accuracy}


dcgan = DCGAN(discriminator=discriminator, generator=generator, latent_dim=LATENT_DIM)
dcgan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=True),
)

In [ ]:
CHECKPOINT_DIR = Path("artifacts/dcgan")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SEED_LATENT = tf.random.normal((16, LATENT_DIM))

class SampleCallback(keras.callbacks.Callback):
    def __init__(self, generator: keras.Model, latent_vectors: tf.Tensor, out_dir: Path):
        self.generator = generator
        self.latent_vectors = latent_vectors
        self.out_dir = out_dir

    def on_epoch_end(self, epoch, logs=None):
        generated = self.generator(self.latent_vectors, training=False)
        generated = (generated + 1) / 2
        grid_cols = 4
        grid_rows = int(np.ceil(len(self.latent_vectors) / grid_cols))
        plt.figure(figsize=(grid_cols * 2, grid_rows * 2))
        for i, img in enumerate(generated):
            plt.subplot(grid_rows, grid_cols, i + 1)
            plt.imshow(img.numpy())
            plt.axis("off")
        plt.suptitle(f"DCGAN: примеры после эпохи {epoch + 1}")
        plt.tight_layout()
        plt.show()
        np.save(self.out_dir / f"epoch_{epoch + 1:03d}.npy", generated.numpy())

EPOCHS = 30
steps_per_epoch = min(500, tf.data.experimental.cardinality(train_images).numpy())

history = dcgan.fit(
    train_images,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=[SampleCallback(generator, SEED_LATENT, CHECKPOINT_DIR)],
)

In [ ]:
def generate_images(generator: keras.Model, num_images: int = 10) -> np.ndarray:
    noise = tf.random.normal((num_images, LATENT_DIM))
    generated = generator(noise, training=False)
    return (generated + 1) / 2

synthetic_batch = generate_images(generator, 10)
plt.figure(figsize=(12, 6))
for idx, img in enumerate(synthetic_batch):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(img.numpy())
    plt.axis("off")
plt.suptitle("10 изображений от DCGAN")
plt.tight_layout()
plt.show()

## Метрика качества: Frechet Inception Distance

FID измеряет расстояние между распределениями признаков реальных и синтетических изображений в пространстве признаков InceptionV3. Чем меньше значение, тем ближе синтетическое распределение к настоящему.

In [ ]:
FID_IMAGE_SIZE = (299, 299)
FID_MODEL = tf.keras.applications.InceptionV3(
    include_top=False, weights="imagenet", pooling="avg", input_shape=FID_IMAGE_SIZE + (3,)
)


def sample_images(ds: tf.data.Dataset, max_images: int = 500) -> np.ndarray:
    collected = []
    for batch in ds:
        collected.append(batch.numpy())
        if sum(x.shape[0] for x in collected) >= max_images:
            break
    images = np.concatenate(collected, axis=0)
    return images[:max_images]


def preprocess_for_fid(images: np.ndarray) -> tf.Tensor:
    images = tf.convert_to_tensor(images)
    images = tf.image.resize(images, FID_IMAGE_SIZE)
    images = (images + 1.0) * 127.5
    return tf.keras.applications.inception_v3.preprocess_input(images)


def calculate_fid(real_images: np.ndarray, fake_images: np.ndarray) -> float:
    real_acts = FID_MODEL(preprocess_for_fid(real_images))
    fake_acts = FID_MODEL(preprocess_for_fid(fake_images))
    mu1, sigma1 = tf.reduce_mean(real_acts, axis=0), tfp.stats.covariance(real_acts)
    mu2, sigma2 = tf.reduce_mean(fake_acts, axis=0), tfp.stats.covariance(fake_acts)
    diff = mu1 - mu2
    covmean = tf.linalg.sqrtm(tf.matmul(sigma1, sigma2))
    if tf.math.is_finite(tf.math.reduce_sum(covmean)) is False:
        eps = tf.eye(sigma1.shape[0]) * 1e-6
        covmean = tf.linalg.sqrtm(tf.matmul(sigma1 + eps, sigma2 + eps))
    fid = tf.reduce_sum(diff * diff) + tf.linalg.trace(sigma1 + sigma2 - 2.0 * covmean)
    return float(tf.math.real(fid))

In [ ]:
real_eval = sample_images(val_images, max_images=300)
fake_eval = generate_images(generator, num_images=300).numpy()
dcgan_fid = calculate_fid(real_eval, fake_eval)
print(f"FID для DCGAN: {dcgan_fid:.2f}")

## Stable Diffusion: генерация изображений

Используем `diffusers` и готовую модель `runwayml/stable-diffusion-v1-5`. Необходимо заранее сохранить токен HuggingFace в переменной окружения `HUGGINGFACE_TOKEN`.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = "runwayml/stable-diffusion-v1-5"
PROMPT = "close-up dermatoscopic photography of pigmented skin lesion, polarized light, medical macro, detailed texture, neutral background"
NEGATIVE_PROMPT = "text, watermark, artifacts, blurry, low contrast"
NUM_SD_IMAGES = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

hf_token = os.getenv("HUGGINGFACE_TOKEN")
if hf_token is None:
    raise EnvironmentError("Задайте переменную окружения HUGGINGFACE_TOKEN")

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    use_auth_token=hf_token,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()

print(f"Pipeline загружен на {DEVICE}")

In [ ]:
sd_images = []
for i in range(NUM_SD_IMAGES):
    image = pipe(
        PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        guidance_scale=7.5,
        num_inference_steps=40,
        height=512,
        width=512,
        generator=torch.Generator(device=DEVICE).manual_seed(SEED + i),
    ).images[0]
    sd_images.append(image)

plt.figure(figsize=(12, 6))
for idx, img in enumerate(sd_images):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(img)
    plt.axis("off")
plt.suptitle("10 изображений от Stable Diffusion")
plt.tight_layout()
plt.show()

In [ ]:
sd_images_arr = np.stack([np.array(img.resize(IMG_SIZE)) for img in sd_images])
sd_images_arr = (sd_images_arr.astype("float32") / 127.5) - 1.0

sd_fid = calculate_fid(real_eval, sd_images_arr)
print(f"FID для Stable Diffusion: {sd_fid:.2f}")

## Сравнение моделей и выводы

- **Точность:** Stable Diffusion достигает более низкого FID (ближе к референсным изображениям) за счёт использования предобученной диффузионной модели. DCGAN демонстрирует более высокое значение метрики, что обусловлено ограниченным объёмом данных и менее выразительной архитектурой.
- **Время:** DCGAN требует длительного обучения (десятки эпох) на каждой новой задаче, тогда как Stable Diffusion готов генерировать изображения сразу после загрузки весов, но каждая генерация может занимать до нескольких секунд на GPU/CPU.
- **Применимость:** DCGAN полезен, когда нужен полный контроль над данными и возможность обучать модель на строго ограниченном домене. Stable Diffusion стоит выбирать, когда необходимо быстро получить качественные изображения или провести data augmentation без долгого обучения.
- **Влияние разрешения и данных:** увеличение разрешения (например, 256×256) резко повышает требования к памяти и времени, поэтому для небольших датасетов разумнее начать с 128×128. Количество эпох и исходных снимков напрямую влияет на разнообразие DCGAN; при недостатке данных полезно применять агрессивные аугментации.

Итого: для расширения датасета и быстрого прототипирования под дерматоскопические снимки целесообразно комбинировать оба подхода: обучать кастомную DCGAN для доменно-специфичных артефактов и дополнять результаты генерациями Stable Diffusion при необходимости повышенного визуального качества.

In [ ]:
comparison_df = pd.DataFrame(
    {
        "Модель": ["DCGAN", "Stable Diffusion"],
        "FID": [dcgan_fid, sd_fid],
        "Комментарии": [
            "Требует обучения с нуля, хорошо контролирует стиль",
            "Использует предобученные веса, даёт высокое качество сразу",
        ],
    }
)
comparison_df